In [3]:
import os
import numpy as np
from scipy.signal import correlate

# Define the folder structure
lengths = ['1m', '5km', '10km']
modulation_schemes = ['16QAM', '32QAM', '64QAM', 'QPSK']

def load_vector(file_path: str) -> np.ndarray:
    """Load a single-column real-valued vector."""
    data = np.loadtxt(file_path)
    return np.atleast_1d(data).astype(float)


def finddelay(sig1, sig2):
    """
    Find the delay between two signals using autocorrelation.
    Returns the number of samples sig2 is delayed relative to sig1.
    """
    # Compute cross-correlation
    correlation = correlate(sig1, sig2, mode='full')
    
    # Find the lag with maximum correlation
    lag = np.arange(-len(sig2) + 1, len(sig1))
    max_idx = np.argmax(np.abs(correlation))
    delay = lag[max_idx]
    
    return delay


def synchronize(rx: np.ndarray, tx: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Align rx to tx using cross-correlation; return trimmed aligned pair."""
    # Use autocorrelation to find the delay
    d = finddelay(tx, rx)
    print(f"  Delay detected: {d} samples")
    
    # Shift for synchronization
    rx_sync = np.roll(rx, d)
    
    # Trim to same length
    min_len = min(len(rx_sync), len(tx))
    return rx_sync[:min_len], tx[:min_len]

# Main processing loop
for length in lengths:
    for scheme in modulation_schemes:
        # Define paths
        rx_folder = os.path.join(os.getcwd(), length, scheme)
        tx_folder = os.path.join(os.getcwd(), 'Tx_Waveform_and_Bits')
        # Load shared TX reference for this scheme
        tx_i_path = os.path.join(tx_folder, f'{scheme}_i.txt')
        tx_q_path = os.path.join(tx_folder, f'{scheme}_q.txt')
        tx_i = load_vector(tx_i_path)
        tx_q = load_vector(tx_q_path)
        tx_waveform = tx_i + 1j * tx_q
        # Get all received waveform files
        rx_files = [f for f in os.listdir(rx_folder) if f.endswith('_i.txt')]
        print(f'Processing {length} - {scheme}, found {len(rx_files)} RX files.')
        
        for rx_file in rx_files:
            # Load received waveforms
            i_rx = load_vector(os.path.join(rx_folder, rx_file))
            q_rx = load_vector(os.path.join(rx_folder, rx_file.replace('_i.txt', '_q.txt')))
            rx_waveform = i_rx + 1j * q_rx

            aligned_rx, aligned_tx = synchronize(rx_waveform, tx_waveform)

            # Pair each row as (rx_complex, tx_complex)
            tx_strings = [f"{c.real:.8f}{c.imag:+.8f}j" for c in aligned_tx]
            rx_strings = [f"{c.real:.8f}{c.imag:+.8f}j" for c in aligned_rx]
            paired = np.column_stack((rx_strings, tx_strings))

            out_name = f'sync_{rx_file.replace("_i.txt", "")}.txt'
            output_file = os.path.join(rx_folder, out_name)
            np.savetxt(output_file, paired, fmt="%s")
            print(f'Saved synchronized data to {output_file}')

print('Synchronization complete!')

Processing 1m - 16QAM, found 10 RX files.
  Delay detected: -2284 samples
Saved synchronized data to /Users/arthurzhao/Desktop/mmWave-AROF-Dataset/1m/16QAM/sync_16QAM_28_3.txt
  Delay detected: -2285 samples
Saved synchronized data to /Users/arthurzhao/Desktop/mmWave-AROF-Dataset/1m/16QAM/sync_16QAM_30_5.txt
  Delay detected: -2284 samples
Saved synchronized data to /Users/arthurzhao/Desktop/mmWave-AROF-Dataset/1m/16QAM/sync_16QAM_29_7.txt
  Delay detected: -2284 samples
Saved synchronized data to /Users/arthurzhao/Desktop/mmWave-AROF-Dataset/1m/16QAM/sync_16QAM_29_5.txt
  Delay detected: -2285 samples
Saved synchronized data to /Users/arthurzhao/Desktop/mmWave-AROF-Dataset/1m/16QAM/sync_16QAM_30_7.txt
  Delay detected: -2285 samples
Saved synchronized data to /Users/arthurzhao/Desktop/mmWave-AROF-Dataset/1m/16QAM/sync_16QAM_28_5.txt
  Delay detected: -2285 samples
Saved synchronized data to /Users/arthurzhao/Desktop/mmWave-AROF-Dataset/1m/16QAM/sync_16QAM_30_3.txt
  Delay detected: -2